<a href="https://colab.research.google.com/github/MarceloCoccaro12/Disciplina_de_Inteligencia_Artificial_UMC/blob/main/FISHMORPH_Order.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!kaggle datasets download -d menegidio/fishmorph-dataset --unzip -p /content/dados

Dataset URL: https://www.kaggle.com/datasets/menegidio/fishmorph-dataset
License(s): MIT
100% 844k/844k [00:00<00:00, 175MB/s]



# Importando Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
fish_db = pd.read_csv('/content/dados/FISHMORPH_Order_Dataset.csv')

In [ ]:
fish_db

,MBl,BEl,VEp,REs,OGp,RMl,BLs,PFv,PFs,CPt,Order
0,130.0,4.579113,0.622642,0.375019,0.622642,0.868659,0.484981,0.273585,0.144108,3.348991,Cypriniformes
1,11.5,2.796068,0.650700,0.353780,0.509800,0.411157,0.722461,0.330169,0.221095,2.525592,Perciformes
2,6.6,4.564329,0.518187,0.312002,0.203985,0.413221,0.591495,0.105517,0.188193,2.104580,Cypriniformes
3,10.9,4.390422,0.548247,0.244804,0.168959,0.324728,0.642991,0.097450,0.193049,1.787062,Cypriniformes
4,18.9,4.410271,0.542353,0.292058,0.183790,0.398567,0.685672,0.087643,0.170648,2.968906,Cypriniformes
...,...,...,...,...,...,...,...,...,...,...,...
8337,22.0,8.160515,0.742393,0.394320,0.307719,0.332283,0.793826,0.600208,0.105301,3.951881,Perciformes
8338,50.0,5.241134,0.600061,0.377792,0.293883,0.301585,0.617588,0.481677,0.133632,3.565657,Perciformes
8339,4.0,3.159920,0.453558,0.560501,0.533168,0.175106,0.470270,0.324433,0.187150,2.395070,Cyprinodontiformes
8340,140.0,4.165031,0.545773,0.115646,0.471200,0.483810,0.498305,0.223729,0.163095,2.091741,Siluriformes


In [ ]:
fish_db.isnull().sum()

,0
MBl,0
BEl,0
VEp,0
REs,0
OGp,0
RMl,0
BLs,0
PFv,0
PFs,0
CPt,0


In [ ]:
fish_db.describe()

,MBl,BEl,VEp,REs,OGp,RMl,BLs,PFv,PFs,CPt
count,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000
mean,22.004517,4.452438,0.549087,0.391665,0.381768,0.399786,0.571818,0.285316,0.189343,2.560795
std,33.612007,2.338235,0.094883,0.132814,0.194352,0.302658,0.117486,0.155629,0.056109,1.016659
min,1.200000,1.033084,0.159248,0.000000,0.000000,0.000000,0.224525,0.000000,0.000000,0.000000
25%,7.000000,3.139102,0.487993,0.293205,0.278014,0.257999,0.490052,0.193134,0.157972,1.999744
50%,12.600000,4.038851,0.546213,0.389429,0.406904,0.361436,0.564739,0.275039,0.183497,2.436329
75%,24.500000,5.002621,0.605125,0.486221,0.516001,0.490671,0.648039,0.368876,0.215031,2.919818
max,800.000000,43.728161,1.000000,1.026667,1.026316,7.060344,1.230105,0.857097,0.511113,15.452061


In [ ]:
fish_db['Order'].unique()

array(['Cypriniformes', 'Perciformes', 'Characiformes', 'Scorpaeniformes',
       'Siluriformes', 'Pleuronectiformes', 'Acipenseriformes',
       'Gymnotiformes', 'Cyprinodontiformes', 'Clupeiformes',
       'Amiiformes', 'Anguilliformes', 'Gasterosteiformes',
       'Percopsiformes', 'Osmeriformes', 'Osteoglossiformes',
       'Gobiesociformes', 'Beloniformes', 'Atheriniformes',
       'Lepisosteiformes', 'Tetraodontiformes', 'Salmoniformes',
       'Gonorynchiformes', 'Synbranchiformes', 'Mugiliformes',
       'Batrachoidiformes', 'Esociformes', 'Syngnathiformes',
       'Gadiformes', 'Elopiformes', 'Polypteriformes', 'Ophidiiformes'],
      dtype=object)

In [ ]:
print("Contagem de repetições por order :\n", fish_db['Order'].value_counts())

Contagem de repetições por order :
 Order
Cypriniformes         1933
Perciformes           1826
Siluriformes          1800
Characiformes         1281
Cyprinodontiformes     554
Osteoglossiformes      156
Clupeiformes           104
Gymnotiformes          101
Salmoniformes           92
Atheriniformes          71
Scorpaeniformes         70
Osmeriformes            69
Synbranchiformes        48
Beloniformes            33
Tetraodontiformes       31
Pleuronectiformes       29
Acipenseriformes        23
Mugiliformes            18
Syngnathiformes         16
Anguilliformes          16
Gasterosteiformes       13
Gonorynchiformes        10
Esociformes             10
Polypteriformes          9
Lepisosteiformes         6
Percopsiformes           6
Elopiformes              5
Batrachoidiformes        4
Gadiformes               3
Gobiesociformes          3
Amiiformes               1
Ophidiiformes            1
Name: count, dtype: int64


In [ ]:
fish_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8342 entries, 0 to 8341
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   MBl     8342 non-null   float64
 1   BEl     8342 non-null   float64
 2   VEp     8342 non-null   float64
 3   REs     8342 non-null   float64
 4   OGp     8342 non-null   float64
 5   RMl     8342 non-null   float64
 6   BLs     8342 non-null   float64
 7   PFv     8342 non-null   float64
 8   PFs     8342 non-null   float64
 9   CPt     8342 non-null   float64
 10  Order   8342 non-null   object 
dtypes: float64(10), object(1)
memory usage: 717.0+ KB


In [ ]:
X = fish_db.drop('Order', axis=1)
y = fish_db['Order']

In [ ]:
numerical_features = ['MBl', 'BEl', 'VEp', 'REs', 'OGp', 'RMl',	'BLs', 'PFv', 'PFs', 'CPt']

In [ ]:
numerical_transformer = StandardScaler()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features)
    ]
)

In [ ]:
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', SVC(kernel='linear'))])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['MBl', 'BEl', 'VEp', 'REs',
                                                   'OGp', 'RMl', 'BLs', 'PFv',
                                                   'PFs', 'CPt'])])),
                ('classifier', SVC(kernel='linear'))])

In [ ]:
y_pred = model_pipeline.predict(X_test)

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
print(f'Acuracia: {acuracia * 100:.2f}%')

Acuracia: 70.40%


In [ ]:
param_grid = {
    'classifier__kernel' : ['linear', 'rbf', 'poly', 'sigmoid'],
    'classifier__C' : [0.1, 1, 10, 100],
    'classifier__gamma' : ['scale', 'auto', 0.1, 1]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    scoring='accuracy',
    n_jobs=-1,
    cv=5,
    verbose=2
)

grid_search.fit(X_train, y_train)

# Obter melhor parâmetro e melhor pontuação de validação cruzada
best_param = grid_search.best_params_
best_score = grid_search.best_score_


print(f'Melhores parâmetros encontrados: {best_param}')
print(f'Melhor Acurácia Média (Validação Cruzada): {best_score}')

Fitting 5 folds for each of 64 candidates, totalling 320 fits


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
model_best = grid_search.best_estimator_
y_pred_tuned = model_best.predict(X_test)
acuracia_teste = accuracy_score(y_test, y_pred_tuned)
print(f'Acurácia do conjunto de teste: {acuracia_teste * 100:.2f}%')

In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned)